# Phase 2 â€” Memory-Based Collaborative Filtering
User-User CF and Item-Item CF on MovieLens-1M.

In [1]:
import sys, time
sys.path.insert(0, '..')
import pandas as pd
import numpy as np
from src.cf import UserUserCF, ItemItemCF
from src.data import load_movies

In [2]:
train = pd.read_csv('../data/train.csv')
val   = pd.read_csv('../data/val.csv')
movies = load_movies()
print(f'train: {train.shape}  val: {val.shape}')
train.head(3)

train: (805443, 4)  val: (97383, 4)


,userId,movieId,rating,timestamp
0,1,3186,4,978300019
1,1,1270,5,978300055
2,1,1721,4,978300055


## 1. User-User CF

In [3]:
t0 = time.time()
uucf = UserUserCF(K=50).fit(train)
print(f'Fit in {time.time()-t0:.1f}s')
print(f'Sim matrix shape: {uucf.sim.shape}')

Fit in 1.9s
Sim matrix shape: (6040, 6040)


In [4]:
# Recommendations for a sample user
USER_ID = 1
recs = uucf.recommend(user_id=USER_ID, n=10)
rec_df = pd.DataFrame(recs, columns=['movieId', 'score'])
rec_df = rec_df.merge(movies[['movieId','title','genres']], on='movieId')
print(f'User-User CF recs for user {USER_ID}:')
rec_df

User-User CF recs for user 1:


,movieId,score,title,genres
0,2208,5.0,"Lady Vanishes, The (1938)",Comedy|Mystery|Romance|Thriller
1,905,5.0,It Happened One Night (1934),Comedy
2,3730,5.0,"Conversation, The (1974)",Drama|Mystery
3,41,5.0,Richard III (1995),Drama|War
4,3951,5.0,Two Family House (2000),Drama
5,944,5.0,Lost Horizon (1937),Drama
6,945,5.0,Top Hat (1935),Comedy|Musical|Romance
7,2618,5.0,"Castle, The (1997)",Comedy
8,928,5.0,Rebecca (1940),Romance|Thriller
9,2351,5.0,Nights of Cabiria (Le Notti di Cabiria) (1957),Drama


In [5]:
# Precision@K on val set (fraction of top-K recs that appear in val)
def precision_at_k(model, val_df, k=10, n_users=200):
    val_items = val_df.groupby('userId')['movieId'].apply(set).to_dict()
    sample_users = list(val_items.keys())[:n_users]
    hits = []
    for uid in sample_users:
        recs = model.recommend(uid, n=k)
        rec_ids = {r[0] for r in recs}
        relevant = val_items.get(uid, set())
        hits.append(len(rec_ids & relevant) / k)
    return np.mean(hits)

p_at_10 = precision_at_k(uucf, val, k=10)
print(f'User-User CF  Precision@10 (first 200 users): {p_at_10:.4f}')

User-User CF  Precision@10 (first 200 users): 0.0015


## 2. Item-Item CF

In [6]:
t0 = time.time()
iicf = ItemItemCF(K=50).fit(train)
print(f'Fit in {time.time()-t0:.1f}s')
print(f'Sim matrix shape: {iicf.sim.shape}')

Fit in 1.4s
Sim matrix shape: (3952, 3952)


In [7]:
recs = iicf.recommend(user_id=USER_ID, n=10)
rec_df2 = pd.DataFrame(recs, columns=['movieId', 'score'])
rec_df2 = rec_df2.merge(movies[['movieId','title','genres']], on='movieId')
print(f'Item-Item CF recs for user {USER_ID}:')
rec_df2

Item-Item CF recs for user 1:


,movieId,score,title,genres
0,1196,1.569880,Star Wars: Episode V - The Empire Strikes Back...,Action|Adventure|Drama|Sci-Fi|War
1,1198,1.505250,Raiders of the Lost Ark (1981),Action|Adventure
2,318,1.484913,"Shawshank Redemption, The (1994)",Drama
3,1,1.476254,Toy Story (1995),Animation|Children's|Comedy
4,1265,1.459898,Groundhog Day (1993),Comedy|Romance
5,593,1.452691,"Silence of the Lambs, The (1991)",Drama|Thriller
6,296,1.433502,Pulp Fiction (1994),Crime|Drama
7,1210,1.433441,Star Wars: Episode VI - Return of the Jedi (1983),Action|Adventure|Romance|Sci-Fi|War
8,2716,1.426277,Ghostbusters (1984),Comedy|Horror
9,2174,1.407108,Beetlejuice (1988),Comedy|Fantasy


In [8]:
p_at_10_ii = precision_at_k(iicf, val, k=10)
print(f'Item-Item CF  Precision@10 (first 200 users): {p_at_10_ii:.4f}')

Item-Item CF  Precision@10 (first 200 users): 0.0855


## 3. Summary

| Model | Precision@10 |
|---|---|
| User-User CF | 0.0015 |
| Item-Item CF | 0.0855 |

Item-Item CF outperforms User-User CF significantly. User-User surfaces niche high-rated films (score=5.0) that do not match the user's actual next watches. Item-Item recommends items similar to the user's history, aligning better with temporal val splits.